# Clase 7 — Criptografía de Curvas Elípticas (ECC)

> **Curso:** Criptografía Aplicada  
> **Objetivo:** Comprender la matemática de las curvas elípticas, su aritmética de grupo, los protocolos ECDH y ECDSA, las curvas estándar más utilizadas (P-256, Curve25519, Ed25519) y sus ventajas de seguridad y eficiencia frente a RSA.

---

## Tabla de contenidos

1. [Introducción y motivación: ¿por qué ECC?](#1)  
2. [Estructuras algebraicas: grupos, anillos y cuerpos](#2)  
   2.1 [Grupos](#2-1)  
   2.2 [Anillos y cuerpos finitos](#2-2)  
   2.3 [El grupo aditivo de una curva elíptica](#2-3)  
3. [Aritmética en curvas elípticas sobre $\mathbb{F}_p$](#3)  
   3.1 [Ecuación de Weierstrass y forma corta](#3-1)  
   3.2 [Punto en el infinito y suma de puntos](#3-2)  
   3.3 [Fórmulas explícitas: suma y duplicado](#3-3)  
   3.4 [Multiplicación escalar (double-and-add)](#3-4)  
   3.5 [Orden del grupo y subgrupos](#3-5)  
4. [Implementación Python desde cero](#4)  
   4.1 [Clase `ECPoint` sobre $\mathbb{F}_p$](#4-1)  
   4.2 [Curva genérica configurable](#4-2)  
   4.3 [Verificación de propiedades del grupo](#4-3)  
5. [Curvas estándar](#5)  
   5.1 [Familia NIST: P-256 (secp256r1)](#5-1)  
   5.2 [secp256k1 (Bitcoin)](#5-2)  
   5.3 [Curve25519 / X25519](#5-3)  
   5.4 [Ed25519 (curvas de Edwards)](#5-4)  
   5.5 [Comparativa de curvas](#5-5)  
6. [ECDH — Intercambio de clave Diffie-Hellman con curvas elípticas](#6)  
   6.1 [Protocolo ECDH paso a paso](#6-1)  
   6.2 [Implementación educativa en Python puro](#6-2)  
   6.3 [ECDH con la biblioteca `cryptography` (X25519)](#6-3)  
   6.4 [Cifrado híbrido ECDH + AES-GCM](#6-4)  
7. [ECDSA — Firma digital](#7)  
   7.1 [Algoritmo de firma y verificación](#7-1)  
   7.2 [Implementación educativa en Python puro](#7-2)  
   7.3 [Fallos catastróficos del nonce $k$](#7-3)  
   7.4 [ECDSA con la biblioteca `cryptography` (P-256)](#7-4)  
8. [EdDSA / Ed25519](#8)  
   8.1 [Curvas de Edwards torsionadas](#8-1)  
   8.2 [Ventajas de Ed25519 sobre ECDSA](#8-2)  
   8.3 [Implementación con `cryptography`](#8-3)  
9. [Ataques y consideraciones de seguridad](#9)  
   9.1 [Problema del logaritmo discreto en EC (ECDLP)](#9-1)  
   9.2 [Ataques genéricos: baby-step giant-step y Pohlig-Hellman](#9-2)  
   9.3 [Curvas débiles: MOV, anomalous, curvas de cofactor alto](#9-3)  
   9.4 [Ataque de reutilización de nonce en ECDSA (Sony PS3)](#9-4)  
   9.5 [Side-channel attacks](#9-5)  
10. [ECC vs RSA: comparativa exhaustiva](#10)  
11. [Buenas prácticas](#11)  
12. [Referencias](#12)

---

## Importaciones globales

In [ ]:
import math, random, hashlib, secrets, os, time, base64
from typing import Optional, Tuple

# Biblioteca de criptografía de producción
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.backends import default_backend

print("Importaciones correctas ✓")

---
<a id='1'></a>
## 1. Introducción y motivación: ¿por qué ECC?

### 1.1 El problema de los tamaños de clave

RSA debe su seguridad a la **factorización de enteros**: dado $n = p \cdot q$, encontrar $p$ y $q$ es computacionalmente difícil para $n$ grande.  
ECC se basa en el **problema del logaritmo discreto en curvas elípticas (ECDLP)**: dado un punto $Q = k \cdot P$ en la curva, encontrar el escalar $k$ es computacionalmente difícil.

El ECDLP es matemáticamente más duro en el sentido de que **no se conocen sub-exponenciales generales** para resolverlo (a diferencia de la factorización, para la que existen el *Number Field Sieve* y el *Index Calculus*). Esto se traduce en claves mucho más pequeñas para el mismo nivel de seguridad:

| Nivel de seguridad | RSA / DH | ECC |
|---|---|---|
| 80 bits | 1024 bits | 160 bits |
| 112 bits | 2048 bits | 224 bits |
| 128 bits | 3072 bits | 256 bits |
| 192 bits | 7680 bits | 384 bits |
| 256 bits | 15360 bits | 521 bits |

*Fuente: NIST SP 800-57 Part 1 Rev. 5*

### 1.2 Impacto práctico

- **Certificados TLS más pequeños** → menos bytes en el handshake → menor latencia.
- **Claves más cortas** → operaciones de firma/verificación más rápidas en hardware limitado (IoT, tarjetas inteligentes).
- **Menor consumo energético** → fundamental en dispositivos embebidos.
- **TLS 1.3** utiliza exclusivamente ECDHE para el intercambio de clave, eliminando RSA-KEX.

### 1.3 Adopción actual

- HTTPS: curvas P-256 y X25519 son las más utilizadas.
- SSH: Ed25519 reemplaza a RSA-2048 como estándar de facto.
- Criptomonedas: secp256k1 (Bitcoin, Ethereum).
- Signal/WhatsApp: Curve25519 para el protocolo X3DH.
- Pasaportes electrónicos: ECDSA con P-256 o brainpoolP256r1.

---
<a id='2'></a>
## 2. Estructuras algebraicas: grupos, anillos y cuerpos

<a id='2-1'></a>
### 2.1 Grupos

Un **grupo** $(G, \star)$ es un conjunto $G$ con una operación binaria $\star$ que satisface:

| Propiedad | Definición |
|---|---|
| Clausura | $\forall\, a, b \in G: a \star b \in G$ |
| Asociatividad | $(a \star b) \star c = a \star (b \star c)$ |
| Identidad | $\exists\, e \in G: a \star e = e \star a = a$ |
| Inverso | $\forall\, a \in G,\, \exists\, a^{-1}: a \star a^{-1} = e$ |

Si además $a \star b = b \star a$ para todo par, el grupo es **abeliano (conmutativo)**.

**Ejemplos:**
- $(\mathbb{Z}, +)$: grupo abeliano infinito.
- $(\mathbb{Z}_n, +)$: grupo abeliano finito de orden $n$.
- $(\mathbb{Z}_p^*, \times)$ con $p$ primo: grupo cíclico de orden $p-1$.
- Puntos de una curva elíptica con la operación de suma: grupo abeliano.

**Orden** de un grupo: número de elementos, denotado $|G|$.

**Generador** $g$: elemento tal que $\{g, g^2, g^3, \ldots\} = G$ (grupo cíclico).

<a id='2-2'></a>
### 2.2 Anillos y cuerpos finitos

Un **cuerpo finito** (o campo de Galois) $\mathbb{F}_q$ con $q = p^m$ ($p$ primo) es un conjunto finito con operaciones de suma y multiplicación bien definidas, donde todo elemento no nulo tiene inverso multiplicativo.

**Tipos relevantes para ECC:**

| Cuerpo | Notación | Uso |
|---|---|---|
| Enteros módulo primo | $\mathbb{F}_p$ | Curvas como P-256, secp256k1 |
| Extensión binaria | $\mathbb{F}_{2^m}$ | Curvas binarias (menos usadas hoy) |

Las curvas más importantes en la práctica operan sobre $\mathbb{F}_p$.

<a id='2-3'></a>
### 2.3 El grupo aditivo de una curva elíptica

Los **puntos** de una curva elíptica, junto con el **punto en el infinito** $\mathcal{O}$ y la operación de suma geométrica, forman un **grupo abeliano finito**.  
Este grupo es el fundamento de toda la criptografía ECC:

- La **multiplicación escalar** $k \cdot P$ (suma de $P$ consigo mismo $k$ veces) es la operación clave.
- Es fácil calcular $Q = k \cdot P$ dado $k$ y $P$.
- Es difícil calcular $k$ dado $Q$ y $P$ → **ECDLP**.

---
<a id='3'></a>
## 3. Aritmética en curvas elípticas sobre $\mathbb{F}_p$

<a id='3-1'></a>
### 3.1 Ecuación de Weierstrass y forma corta

La **ecuación de Weierstrass generalizada** es:

$$y^2 + a_1 xy + a_3 y = x^3 + a_2 x^2 + a_4 x + a_6$$

Para cuerpos de característica $\neq 2, 3$ (es decir, la mayoría de $\mathbb{F}_p$ con $p > 3$), se puede simplificar a la **forma corta de Weierstrass**:

$$\boxed{y^2 = x^3 + ax + b \pmod{p}}$$

**Condición de no singularidad** (la curva debe ser suave):

$$\Delta = -16(4a^3 + 27b^2) \neq 0 \pmod{p}$$

Si $\Delta = 0$, la curva tiene un punto singular (cúspide o nodo) y **no** forma un grupo.

**Ejemplo — curva juguete sobre $\mathbb{F}_{23}$:**

$$E: y^2 = x^3 + x + 1 \pmod{23} \quad (a=1, b=1)$$

<a id='3-2'></a>
### 3.2 Punto en el infinito y suma de puntos

El **punto en el infinito** $\mathcal{O}$ es el **elemento neutro** del grupo.  
Geométricamente, $\mathcal{O}$ es el punto donde se cortan todas las rectas verticales en el plano proyectivo.

**Ley de grupo (geometría):**

Para sumar $P + Q$:
1. Trazar la recta que pasa por $P$ y $Q$ (o la tangente si $P = Q$).
2. Encontrar el tercer punto de intersección $R'$ de esa recta con la curva.
3. Reflejar $R'$ respecto al eje $x$: el resultado es $R = P + Q$.

**Casos especiales:**
- $P + \mathcal{O} = \mathcal{O} + P = P$ (identidad).
- $P + (-P) = \mathcal{O}$, donde $-P = (x, -y \bmod p)$.
- Si $P = Q$ y $y_P = 0$: la tangente es vertical, resultado $= \mathcal{O}$.

<a id='3-3'></a>
### 3.3 Fórmulas explícitas: suma y duplicado

**Suma de dos puntos distintos** $P = (x_1, y_1)$, $Q = (x_2, y_2)$, $P \neq Q$:

$$\lambda = \frac{y_2 - y_1}{x_2 - x_1} \pmod{p}$$
$$x_3 = \lambda^2 - x_1 - x_2 \pmod{p}$$
$$y_3 = \lambda(x_1 - x_3) - y_1 \pmod{p}$$

**Duplicado de un punto** $P + P = 2P$ (tangente en $P$):

$$\lambda = \frac{3x_1^2 + a}{2y_1} \pmod{p}$$
$$x_3 = \lambda^2 - 2x_1 \pmod{p}$$
$$y_3 = \lambda(x_1 - x_3) - y_1 \pmod{p}$$

<a id='3-4'></a>
### 3.4 Multiplicación escalar (double-and-add)

La multiplicación escalar $k \cdot P$ es el análogo de la exponenciación modular en RSA.  
El algoritmo **double-and-add** tiene complejidad $O(\log k)$:

```
R ← O  (punto en el infinito)
Repetir para cada bit de k (de mayor a menor):
    R ← 2·R
    Si el bit actual es 1:
        R ← R + P
Devolver R
```

<a id='3-5'></a>
### 3.5 Orden del grupo y subgrupos

**Teorema de Hasse:** El número de puntos $|E(\mathbb{F}_p)|$ satisface:

$$|p + 1 - |E(\mathbb{F}_p)|| \leq 2\sqrt{p}$$

En la práctica, para criptografía se eligen curvas donde $|E(\mathbb{F}_p)| = h \cdot n$:
- $n$ es un **primo grande** (el orden del subgrupo generado por el punto base $G$).
- $h$ es el **cofactor** (típicamente $h = 1, 2,$ o $4$).

El punto base $G$ genera el subgrupo criptográfico de orden $n$.

---
<a id='4'></a>
## 4. Implementación Python desde cero

<a id='4-1'></a>
### 4.1 Clase `ECPoint` sobre $\mathbb{F}_p$

In [ ]:
class EllipticCurve:
    """Curva elíptica en forma corta de Weierstrass: y^2 = x^3 + ax + b  (mod p)."""

    def __init__(self, a: int, b: int, p: int):
        self.a = a % p
        self.b = b % p
        self.p = p
        # Verificar que la curva no es singular
        disc = (-16 * (4 * pow(a, 3, p) + 27 * pow(b, 2, p))) % p
        if disc == 0:
            raise ValueError("Curva singular (discriminante = 0). Elige otros parámetros.")

    def is_on_curve(self, x: int, y: int) -> bool:
        """Comprueba que (x, y) satisface la ecuación de la curva."""
        lhs = pow(y, 2, self.p)
        rhs = (pow(x, 3, self.p) + self.a * x + self.b) % self.p
        return lhs == rhs

    def __repr__(self):
        return f"y² ≡ x³ + {self.a}x + {self.b}  (mod {self.p})"


class ECPoint:
    """Punto en una curva elíptica. None representa el punto en el infinito O."""

    def __init__(self, curve: EllipticCurve, x: Optional[int], y: Optional[int]):
        self.curve = curve
        if x is None and y is None:
            # Punto en el infinito
            self.x = self.y = None
        else:
            assert curve.is_on_curve(x, y), f"El punto ({x}, {y}) no está en la curva."
            self.x = x % curve.p
            self.y = y % curve.p

    def is_infinity(self) -> bool:
        return self.x is None

    def __neg__(self) -> "ECPoint":
        """Negación: -(x, y) = (x, -y mod p)."""
        if self.is_infinity():
            return self
        return ECPoint(self.curve, self.x, (-self.y) % self.curve.p)

    def __eq__(self, other: "ECPoint") -> bool:
        if not isinstance(other, ECPoint):
            return False
        if self.is_infinity() and other.is_infinity():
            return True
        if self.is_infinity() or other.is_infinity():
            return False
        return self.x == other.x and self.y == other.y

    def __add__(self, other: "ECPoint") -> "ECPoint":
        """Suma de puntos con ley de grupo."""
        curve, p = self.curve, self.curve.p
        # Identidad
        if self.is_infinity():
            return other
        if other.is_infinity():
            return self
        # Inversos: P + (-P) = O
        if self.x == other.x:
            if self.y != other.y or self.y == 0:
                return ECPoint(curve, None, None)  # infinito
            # Duplicado
            lam_num = (3 * pow(self.x, 2, p) + curve.a) % p
            lam_den = pow(2 * self.y, -1, p)
        else:
            # Suma ordinaria
            lam_num = (other.y - self.y) % p
            lam_den = pow((other.x - self.x) % p, -1, p)

        lam = (lam_num * lam_den) % p
        x3 = (pow(lam, 2, p) - self.x - other.x) % p
        y3 = (lam * (self.x - x3) - self.y) % p
        return ECPoint(curve, x3, y3)

    def __rmul__(self, k: int) -> "ECPoint":
        """Multiplicación escalar k·P con double-and-add."""
        return self.__mul__(k)

    def __mul__(self, k: int) -> "ECPoint":
        """Multiplicación escalar k·P."""
        if k < 0:
            return (-self).__mul__(-k)
        result = ECPoint(self.curve, None, None)  # O
        addend = self
        while k:
            if k & 1:
                result = result + addend
            addend = addend + addend   # duplicado
            k >>= 1
        return result

    def __repr__(self):
        if self.is_infinity():
            return "ECPoint(O)  [punto en el infinito]"
        return f"ECPoint({self.x}, {self.y})"


print("Clase EllipticCurve y ECPoint definidas ✓")

<a id='4-2'></a>
### 4.2 Curva genérica configurable

Usaremos la curva de ejemplo $E: y^2 = x^3 + x + 1 \pmod{23}$.

In [ ]:
# ── Curva de juguete: y^2 = x^3 + x + 1  (mod 23) ──────────────────────────
curve_toy = EllipticCurve(a=1, b=1, p=23)
print("Curva:", curve_toy)

# Encontrar todos los puntos afines de la curva
def all_points(curve: EllipticCurve):
    points = []
    p = curve.p
    for x in range(p):
        rhs = (pow(x, 3, p) + curve.a * x + curve.b) % p
        for y in range(p):
            if pow(y, 2, p) == rhs:
                points.append((x, y))
    return points

pts = all_points(curve_toy)
print(f"\nNúmero de puntos afines: {len(pts)}")
print(f"Número total (con O):    {len(pts) + 1}")
print("\nPrimeros 10 puntos:", pts[:10])

In [ ]:
# ── Operaciones aritméticas elementales ─────────────────────────────────────
P = ECPoint(curve_toy, 3, 10)
Q = ECPoint(curve_toy, 9, 7)

print("P  =", P)
print("Q  =", Q)
print("P+Q=", P + Q)
print("2P =", P + P)
print("-P =", -P)
print("P + (-P) =", P + (-P))  # debería ser O
print()
print("5·P =", 5 * P)
print("10·P=", 10 * P)

<a id='4-3'></a>
### 4.3 Verificación de propiedades del grupo

In [ ]:
# ── Verificar que E(F_p) es un grupo abeliano ────────────────────────────────
O = ECPoint(curve_toy, None, None)   # elemento neutro

# 1. Elemento neutro
assert P + O == P, "Falla: P + O ≠ P"
assert O + P == P, "Falla: O + P ≠ P"
print("✓ Elemento neutro: P + O = O + P = P")

# 2. Inverso
assert P + (-P) == O, "Falla: P + (-P) ≠ O"
print("✓ Inverso: P + (-P) = O")

# 3. Conmutatividad
assert P + Q == Q + P, "Falla: P + Q ≠ Q + P"
print("✓ Conmutatividad: P + Q = Q + P")

# 4. Asociatividad
R = ECPoint(curve_toy, 1, 7)   # un tercer punto
assert (P + Q) + R == P + (Q + R), "Falla: (P+Q)+R ≠ P+(Q+R)"
print("✓ Asociatividad: (P+Q)+R = P+(Q+R)")

# 5. Multiplicación escalar coherente con suma repetida
assert 4 * P == P + P + P + P, "Falla: 4P ≠ P+P+P+P"
print("✓ Multiplicación escalar: 4·P = P+P+P+P")

print("\nTodas las propiedades de grupo abeliano verificadas ✓")

---
<a id='5'></a>
## 5. Curvas estándar

En la práctica, **nunca se deben elegir parámetros de curva arbitrarios**. Se usan curvas estandarizadas con propiedades de seguridad auditadas.

<a id='5-1'></a>
### 5.1 Familia NIST: P-256 (secp256r1 / prime256v1)

Definida en FIPS 186-5.  
Operación: $y^2 = x^3 - 3x + b \pmod{p}$

- $p = 2^{256} - 2^{224} + 2^{192} + 2^{96} - 1$ (primo de Mersenne generalizado)
- Cofactor $h = 1$, orden $n$ primo
- Usada en TLS, certificados X.509, FIDO2/WebAuthn

**Crítica:** los parámetros fueron generados por la NSA (semilla *nada-up-my-sleeve* no verificable → controversia *dual EC DRBG*).

<a id='5-2'></a>
### 5.2 secp256k1 (Bitcoin)

$y^2 = x^3 + 7 \pmod{p}$ con $p = 2^{256} - 2^{32} - 977$.

- $a = 0$, $b = 7$: estructura especial que permite optimizaciones (endomorphism).
- Usada en Bitcoin, Ethereum y la mayoría de criptomonedas.
- Parámetros generados de forma verificable (más transparente que NIST).

<a id='5-3'></a>
### 5.3 Curve25519 / X25519

Diseñada por Bernstein (2006). Curva de Montgomery:

$$y^2 = x^3 + 486662 \cdot x^2 + x \pmod{2^{255} - 19}$$

- Cofactor $h = 8$; orden del subgrupo prima $n \approx 2^{252}$.
- **Resistente a side-channel** por diseño (no requiere comprobación de punto en curva en el destinatario).
- **X25519**: función de intercambio de clave DH usando solo la coordenada $x$.
- Usada en TLS 1.3, SSH, Signal, WireGuard.

<a id='5-4'></a>
### 5.4 Ed25519 (curvas de Edwards)

Curva de Edwards torsionada sobre $\mathbb{F}_{2^{255}-19}$:

$$-x^2 + y^2 = 1 - \frac{121665}{121666} x^2 y^2$$

- Firma determinista (sin nonce aleatorio).
- Muy rápida: ~20 000 firmas/seg en hardware moderno.
- Usada en SSH, GPG, Tor, Monero, Solana.

<a id='5-5'></a>
### 5.5 Comparativa de curvas

In [ ]:
# Parámetros de las curvas estándar más importantes (solo fines educativos)
curves_info = [
    {
        "nombre": "P-256 (secp256r1)",
        "tipo": "Short Weierstrass",
        "primo": "2^256 - 2^224 + 2^192 + 2^96 - 1",
        "bits_seguridad": 128,
        "cofactor": 1,
        "usos": "TLS, X.509, FIDO2",
        "organismo": "NIST",
    },
    {
        "nombre": "P-384 (secp384r1)",
        "tipo": "Short Weierstrass",
        "primo": "2^384 - 2^128 - 2^96 + 2^32 - 1",
        "bits_seguridad": 192,
        "cofactor": 1,
        "usos": "TLS, firmware signing",
        "organismo": "NIST",
    },
    {
        "nombre": "secp256k1",
        "tipo": "Short Weierstrass",
        "primo": "2^256 - 2^32 - 977",
        "bits_seguridad": 128,
        "cofactor": 1,
        "usos": "Bitcoin, Ethereum",
        "organismo": "SECG",
    },
    {
        "nombre": "Curve25519 / X25519",
        "tipo": "Montgomery",
        "primo": "2^255 - 19",
        "bits_seguridad": 128,
        "cofactor": 8,
        "usos": "TLS 1.3, SSH, Signal, WireGuard",
        "organismo": "Bernstein",
    },
    {
        "nombre": "Ed25519",
        "tipo": "Edwards torsionada",
        "primo": "2^255 - 19",
        "bits_seguridad": 128,
        "cofactor": 8,
        "usos": "SSH, GPG, Tor, Solana",
        "organismo": "Bernstein",
    },
]

print(f"{'Curva':<22} {'Tipo':<22} {'Seg. bits':>9} {'Cofactor':>8}  {'Usos':<30} {'Org':<10}")
print("-" * 100)
for c in curves_info:
    print(
        f"{c['nombre']:<22} {c['tipo']:<22} {c['bits_seguridad']:>9} "
        f"{c['cofactor']:>8}  {c['usos']:<30} {c['organismo']:<10}"
    )

---
<a id='6'></a>
## 6. ECDH — Intercambio de clave Diffie-Hellman con curvas elípticas

<a id='6-1'></a>
### 6.1 Protocolo ECDH paso a paso

ECDH es el análogo de Diffie-Hellman clásico, pero usando la multiplicación escalar en curvas elípticas en lugar de la exponenciación modular.

**Parámetros públicos del dominio:** $(E, p, G, n, h)$
- $E$: curva elíptica
- $p$: primo del cuerpo
- $G$: punto generador base
- $n$: orden del subgrupo (primo)
- $h$: cofactor

**Protocolo:**

| | Alice | Bob |
|---|---|---|
| Clave privada | $a \xleftarrow{R} [1, n-1]$ | $b \xleftarrow{R} [1, n-1]$ |
| Clave pública | $A = a \cdot G$ | $B = b \cdot G$ |
| Intercambio | Envía $A$ a Bob | Envía $B$ a Alice |
| Secreto compartido | $S = a \cdot B = a \cdot b \cdot G$ | $S = b \cdot A = b \cdot a \cdot G$ |

$a \cdot b \cdot G = b \cdot a \cdot G$ porque el grupo es **abeliano**.

Eva (atacante pasivo) conoce $G$, $A$, $B$, pero calcular $a$ o $b$ requiere resolver el ECDLP.

**ECDHE (Ephemeral):** Las claves $a$, $b$ se generan de forma efímera para cada sesión, logrando **Perfect Forward Secrecy (PFS)**.

<a id='6-2'></a>
### 6.2 Implementación educativa en Python puro

In [ ]:
# ── Curva de ejemplo de mayor tamaño para ECDH educativo ────────────────────
# Usamos una curva pequeña pero con orden primo para demostrar el protocolo.
# NOTA: Esta curva NO es segura; solo sirve para ilustración.

# Parámetros de la curva: y^2 = x^3 + 2x + 3  (mod 97)
# (curva con orden primo para el punto base elegido)
curve_demo = EllipticCurve(a=2, b=3, p=97)

# Encontrar el orden del grupo con el punto base G = (3, 6)
G_demo = ECPoint(curve_demo, 3, 6)
assert curve_demo.is_on_curve(3, 6), "G no está en la curva"

def point_order(P: ECPoint) -> int:
    """Calcula el orden de un punto (solo para curvas pequeñas)."""
    Q = P
    order = 1
    while not Q.is_infinity():
        Q = Q + P
        order += 1
    return order

n_demo = point_order(G_demo)
print(f"Curva: {curve_demo}")
print(f"Punto base G = {G_demo}")
print(f"Orden de G:  n = {n_demo}  ({'primo' if all(n_demo % i != 0 for i in range(2, int(n_demo**0.5)+1)) else 'compuesto'})")
print()

# ── Protocolo ECDH ──────────────────────────────────────────────────────────
random.seed(42)

# Alice
a_priv = random.randint(2, n_demo - 1)   # clave privada Alice
A_pub  = a_priv * G_demo                 # clave pública Alice

# Bob
b_priv = random.randint(2, n_demo - 1)   # clave privada Bob
B_pub  = b_priv * G_demo                 # clave pública Bob

print(f"Alice: clave privada a = {a_priv}")
print(f"Alice: clave pública  A = {A_pub}")
print(f"Bob  : clave privada b = {b_priv}")
print(f"Bob  : clave pública  B = {B_pub}")
print()

# Secreto compartido
S_alice = a_priv * B_pub
S_bob   = b_priv * A_pub

print(f"Secreto Alice (a·B): {S_alice}")
print(f"Secreto Bob   (b·A): {S_bob}")
print(f"\nSecretos iguales: {S_alice == S_bob} ✓")

<a id='6-3'></a>
### 6.3 ECDH con la biblioteca `cryptography` (X25519)

In [ ]:
# ── ECDH / X25519 con la biblioteca cryptography ─────────────────────────────

# Generar pares de claves
alice_priv = X25519PrivateKey.generate()
bob_priv   = X25519PrivateKey.generate()

alice_pub = alice_priv.public_key()
bob_pub   = bob_priv.public_key()

# Intercambio de clave
shared_alice = alice_priv.exchange(bob_pub)
shared_bob   = bob_priv.exchange(alice_pub)

assert shared_alice == shared_bob, "¡Los secretos no coinciden!"
print(f"Secreto compartido (hex): {shared_alice.hex()}")
print(f"Longitud: {len(shared_alice)} bytes ({len(shared_alice)*8} bits)")
print("Secretos idénticos ✓")

# Serialización de clave pública (formato raw 32 bytes)
alice_pub_bytes = alice_pub.public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw
)
print(f"\nClave pública Alice (raw): {alice_pub_bytes.hex()}")

<a id='6-4'></a>
### 6.4 Cifrado híbrido ECDH + AES-GCM

ECDH solo produce un secreto compartido. Para cifrar datos reales se combina con un KDF y un cifrado simétrico (cifrado híbrido).

In [ ]:
def ecdh_aes_gcm_encrypt(recipient_pub_key, plaintext: bytes) -> dict:
    """
    Cifrado híbrido: X25519 + HKDF + AES-256-GCM.
    Devuelve un dict con los datos necesarios para descifrar.
    """
    # 1. Generar par efímero del emisor
    eph_priv = X25519PrivateKey.generate()
    eph_pub  = eph_priv.public_key()

    # 2. ECDH: secreto compartido
    shared_secret = eph_priv.exchange(recipient_pub_key)

    # 3. KDF: derivar clave AES-256 desde el secreto
    eph_pub_bytes = eph_pub.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw
    )
    aes_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b"ecdh-aes-gcm-v1",
        backend=default_backend()
    ).derive(shared_secret)

    # 4. Cifrar con AES-256-GCM
    nonce = os.urandom(12)  # 96 bits
    aesgcm = AESGCM(aes_key)
    ciphertext = aesgcm.encrypt(nonce, plaintext, associated_data=None)

    return {
        "eph_pub": eph_pub_bytes,   # clave pública efímera (32 bytes)
        "nonce": nonce,
        "ciphertext": ciphertext,
    }


def ecdh_aes_gcm_decrypt(recipient_priv_key, payload: dict) -> bytes:
    """
    Descifrado del paquete generado por ecdh_aes_gcm_encrypt.
    """
    from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PublicKey

    eph_pub_key = X25519PublicKey.from_public_bytes(payload["eph_pub"])
    shared_secret = recipient_priv_key.exchange(eph_pub_key)

    aes_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b"ecdh-aes-gcm-v1",
        backend=default_backend()
    ).derive(shared_secret)

    aesgcm = AESGCM(aes_key)
    return aesgcm.decrypt(payload["nonce"], payload["ciphertext"], associated_data=None)


# ── Demo ─────────────────────────────────────────────────────────────────────
bob_priv_key = X25519PrivateKey.generate()
bob_pub_key  = bob_priv_key.public_key()

mensaje = b"Hola Bob, este mensaje es cifrado con ECDH + AES-GCM. Nadie mas puede leerlo."

payload = ecdh_aes_gcm_encrypt(bob_pub_key, mensaje)
recuperado = ecdh_aes_gcm_decrypt(bob_priv_key, payload)

print("Mensaje original :", mensaje.decode())
print("Clave pública eph:", payload['eph_pub'].hex())
print("Nonce           :", payload['nonce'].hex())
print("Ciphertext      :", payload['ciphertext'].hex())
print("\nMensaje recuperado:", recuperado.decode())
print("Descifrado correcto ✓", mensaje == recuperado)

---
<a id='7'></a>
## 7. ECDSA — Firma digital

<a id='7-1'></a>
### 7.1 Algoritmo de firma y verificación

ECDSA es el estándar de firma digital basado en curvas elípticas (FIPS 186-5, RFC 6979).

**Parámetros de dominio:** $(E, G, n)$ — los mismos que ECDH.

**Generación de claves:**
- Clave privada: $d \xleftarrow{R} [1, n-1]$
- Clave pública: $Q = d \cdot G$

**Firma de un mensaje $m$:**
1. Calcular $z = H(m)$ (hash del mensaje, truncado a $\min(|z|, |n|)$ bits).
2. Generar nonce criptográfico $k \xleftarrow{R} [1, n-1]$ (¡**único por firma**!).
3. Calcular $R = k \cdot G$; sea $r = x_R \bmod n$. Si $r = 0$, elegir otro $k$.
4. Calcular $s = k^{-1}(z + r \cdot d) \bmod n$. Si $s = 0$, elegir otro $k$.
5. Firma: $(r, s)$.

**Verificación de $(r, s)$ sobre $m$ con clave pública $Q$:**
1. Verificar $1 \leq r, s \leq n-1$.
2. Calcular $z = H(m)$.
3. Calcular $w = s^{-1} \bmod n$.
4. Calcular $u_1 = z \cdot w \bmod n$ y $u_2 = r \cdot w \bmod n$.
5. Calcular $X = u_1 \cdot G + u_2 \cdot Q$.
6. Verificar que $x_X \bmod n = r$.

**¿Por qué funciona?**

$$X = u_1 G + u_2 Q = (zw)G + (rw)(dG) = w(z + rd)G = \frac{z + rd}{s} G = \frac{z + rd}{k^{-1}(z + rd)} G = k \cdot G = R$$

<a id='7-2'></a>
### 7.2 Implementación educativa en Python puro

In [ ]:
# ── ECDSA educativo sobre curva pequeña ──────────────────────────────────────
# Reutilizamos curve_demo y G_demo del apartado 6

class ECDSA_Toy:
    """ECDSA simplificado sobre una curva elíptica pequeña."""

    def __init__(self, curve: EllipticCurve, G: ECPoint, n: int):
        self.curve = curve
        self.G = G
        self.n = n   # orden del subgrupo

    def keygen(self) -> Tuple[int, ECPoint]:
        """Genera (clave_privada, clave_pública)."""
        d = secrets.randbelow(self.n - 1) + 1
        Q = d * self.G
        return d, Q

    def _hash_int(self, message: bytes) -> int:
        """Hash del mensaje, reducido módulo n."""
        h = int.from_bytes(hashlib.sha256(message).digest(), 'big')
        return h % self.n

    def sign(self, d: int, message: bytes) -> Tuple[int, int]:
        """Firma el mensaje con la clave privada d."""
        z = self._hash_int(message)
        n = self.n
        while True:
            k = secrets.randbelow(n - 1) + 1
            R = k * self.G
            r = R.x % n
            if r == 0:
                continue
            s = (pow(k, -1, n) * (z + r * d)) % n
            if s != 0:
                return r, s

    def verify(self, Q: ECPoint, message: bytes, r: int, s: int) -> bool:
        """Verifica la firma (r, s) sobre message con la clave pública Q."""
        n = self.n
        if not (1 <= r < n and 1 <= s < n):
            return False
        z = self._hash_int(message)
        w = pow(s, -1, n)
        u1 = (z * w) % n
        u2 = (r * w) % n
        X = u1 * self.G + u2 * Q
        if X.is_infinity():
            return False
        return (X.x % n) == r


# ── Demo ─────────────────────────────────────────────────────────────────────
# Curva con orden primo 199: y^2 = x^3 + 7 (mod 211)
curve_ecdsa = EllipticCurve(a=0, b=7, p=211)
G_ecdsa = ECPoint(curve_ecdsa, 3, 33)
n_ecdsa = 199  # orden primo

ecdsa_toy = ECDSA_Toy(curve_ecdsa, G_ecdsa, n_ecdsa)

d_alice, Q_alice = ecdsa_toy.keygen()
print(f"Clave privada d = {d_alice}")
print(f"Clave pública  Q = {Q_alice}")

msg = b"El saldo de mi cuenta es 9999 euros"
r, s = ecdsa_toy.sign(d_alice, msg)
print(f"\nFirma sobre '{msg.decode()}'")
print(f"  r = {r}")
print(f"  s = {s}")

valido = ecdsa_toy.verify(Q_alice, msg, r, s)
print(f"\nVerificación firma correcta: {valido} ✓")

# Mensaje modificado → firma inválida
msg_falso = b"El saldo de mi cuenta es 0 euros"
valido_falso = ecdsa_toy.verify(Q_alice, msg_falso, r, s)
print(f"Verificación mensaje alterado: {valido_falso} (esperado: False) ✓")

<a id='7-3'></a>
### 7.3 Fallos catastróficos del nonce $k$

El nonce $k$ en ECDSA es **el talón de Aquiles** del esquema. Cualquier fallo en su generación permite recuperar la clave privada.

#### Caso 1: Reutilización de nonce ($k$ repetido en dos firmas)

Si el mismo $k$ se usa para firmar dos mensajes distintos $m_1$ y $m_2$:

$$s_1 = k^{-1}(z_1 + r \cdot d) \bmod n$$
$$s_2 = k^{-1}(z_2 + r \cdot d) \bmod n$$

Como $r$ es el mismo (mismo $k$, mismo $R = kG$), restando:

$$s_1 - s_2 = k^{-1}(z_1 - z_2) \bmod n \implies k = \frac{z_1 - z_2}{s_1 - s_2} \bmod n$$

Recuperado $k$, la clave privada $d$ es inmediata:

$$d = r^{-1}(k \cdot s_1 - z_1) \bmod n$$

**Caso real: Sony PlayStation 3 (2010)** — RPCS3, *fail0verflow*: Sony usó el mismo $k$ para firmar todo su software; los atacantes extrajeron la clave privada y pudieron firmar código arbitrario.

In [ ]:
# ── Demostración: recuperar la clave privada con nonce reutilizado ───────────

def ecdsa_sign_with_k(d: int, message: bytes, k: int, G: ECPoint, n: int) -> Tuple[int, int]:
    """Firma con nonce k fijo (INSEGURO — solo para demostración)."""
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big') % n
    R = k * G
    r = R.x % n
    s = (pow(k, -1, n) * (z + r * d)) % n
    return r, s


def recover_privkey_from_nonce_reuse(
    r: int, s1: int, s2: int, z1: int, z2: int, n: int
) -> int:
    """Recupera la clave privada d si el nonce k fue reutilizado en dos firmas."""
    # k = (z1 - z2) / (s1 - s2) mod n
    diff_s = (s1 - s2) % n
    diff_z = (z1 - z2) % n
    k = (diff_z * pow(diff_s, -1, n)) % n
    # d = (s1*k - z1) / r  mod n
    d = (pow(r, -1, n) * (s1 * k - z1)) % n
    return d


# Setup
d_victim = 47   # clave privada víctima
Q_victim = d_victim * G_ecdsa

msg1 = b"Transaccion #001"
msg2 = b"Transaccion #002"
k_reused = 23   # ¡Nonce reutilizado!

r, s1 = ecdsa_sign_with_k(d_victim, msg1, k_reused, G_ecdsa, n_ecdsa)
_, s2 = ecdsa_sign_with_k(d_victim, msg2, k_reused, G_ecdsa, n_ecdsa)

z1 = int.from_bytes(hashlib.sha256(msg1).digest(), 'big') % n_ecdsa
z2 = int.from_bytes(hashlib.sha256(msg2).digest(), 'big') % n_ecdsa

d_recovered = recover_privkey_from_nonce_reuse(r, s1, s2, z1, z2, n_ecdsa)

print(f"Clave privada víctima  : {d_victim}")
print(f"Clave privada recuperada: {d_recovered}")
print(f"Ataque exitoso         : {d_victim == d_recovered} ✓")
print()
print("MORALEJA: Generar k con secrets.randbelow(n) o usar RFC 6979 (k determinista).")

<a id='7-4'></a>
### 7.4 ECDSA con la biblioteca `cryptography` (P-256)

In [ ]:
# ── ECDSA de producción con P-256 ────────────────────────────────────────────

# Generación de claves
private_key = ec.generate_private_key(ec.SECP256R1())
public_key  = private_key.public_key()

# Serialización
pub_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)
priv_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)
print(pub_pem.decode())

# Firma
mensaje_firma = b"Autorizo la transferencia de 5000 EUR a cuenta ES91 1111 2222 3333"
firma = private_key.sign(mensaje_firma, ec.ECDSA(hashes.SHA256()))
print(f"Firma DER ({len(firma)} bytes): {firma.hex()[:64]}...")

# Verificación
try:
    public_key.verify(firma, mensaje_firma, ec.ECDSA(hashes.SHA256()))
    print("\nFirma verificada correctamente ✓")
except InvalidSignature:
    print("Firma inválida ✗")

# Verificación con mensaje alterado
try:
    public_key.verify(firma, b"Mensaje adulterado", ec.ECDSA(hashes.SHA256()))
except InvalidSignature:
    print("Mensaje adulterado detectado correctamente ✓")

---
<a id='8'></a>
## 8. EdDSA / Ed25519

<a id='8-1'></a>
### 8.1 Curvas de Edwards torsionadas

Las **curvas de Edwards** tienen la forma:

$$x^2 + y^2 = 1 + d \cdot x^2 y^2$$

Las **curvas de Edwards torsionadas** generalizan esto:

$$a \cdot x^2 + y^2 = 1 + d \cdot x^2 y^2$$

Ed25519 usa la curva de Edwards torsionada sobre $\mathbb{F}_{2^{255}-19}$ con $a = -1$.  
Su equivalente en forma de Montgomery es Curve25519.

**Ventajas de las leyes de adición en curvas de Edwards:**
- **Ley de adición unificada**: la misma fórmula para sumar dos puntos distintos *y* para duplicar un punto → elimina ataques de canal lateral basados en ramas.
- Operaciones más rápidas que en forma de Weierstrass corta.

<a id='8-2'></a>
### 8.2 Ventajas de Ed25519 sobre ECDSA

| Característica | ECDSA (P-256) | Ed25519 |
|---|---|---|
| Nonce | Aleatorio (crítico) | **Determinista** (SHA-512 de clave+mensaje) |
| Seguridad frente a RNG débil | Vulnerable | **Inmune** |
| Velocidad de firma | ~1 ms | ~0.05 ms |
| Velocidad de verificación | ~2 ms | ~0.1 ms |
| Tamaño de firma | 71-72 bytes (DER) | **64 bytes** (fijo) |
| Tamaño de clave pública | 65 bytes | **32 bytes** |
| Maleabilidad de firma | Posible en algunas implementaciones | **No maleable** |
| Estandarización | FIPS 186, RFC 6979 | RFC 8032 |

**Firma determinista:** $k$ se genera como $k = H(b \| m)$ donde $b$ es la segunda mitad de la clave privada expandida. Esto hace imposible el ataque de nonce reutilizado.

<a id='8-3'></a>
### 8.3 Implementación con `cryptography`

In [ ]:
# ── Ed25519 ───────────────────────────────────────────────────────────────────

# Generación de claves Ed25519
ed25519_priv = Ed25519PrivateKey.generate()
ed25519_pub  = ed25519_priv.public_key()

# Serialización de clave pública (32 bytes)
pub_raw = ed25519_pub.public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw
)
print(f"Clave pública Ed25519 ({len(pub_raw)} bytes): {pub_raw.hex()}")

# Firma (64 bytes fijos)
mensaje_ed = b"Voto electronico: candidato A"
firma_ed = ed25519_priv.sign(mensaje_ed)
print(f"Firma Ed25519 ({len(firma_ed)} bytes): {firma_ed.hex()}")

# Verificación — NO requiere especificar el algoritmo hash (está implícito)
try:
    ed25519_pub.verify(firma_ed, mensaje_ed)
    print("\nFirma Ed25519 verificada ✓")
except InvalidSignature:
    print("Firma inválida ✗")

# Comparativa de tamaños
p256_priv  = ec.generate_private_key(ec.SECP256R1())
p256_firma = p256_priv.sign(mensaje_ed, ec.ECDSA(hashes.SHA256()))
p256_pub_bytes = p256_priv.public_key().public_bytes(
    serialization.Encoding.X962,
    serialization.PublicFormat.UncompressedPoint
)

print("\n=== Comparativa de tamaños ===")
print(f"  P-256  firma        : {len(p256_firma):>3} bytes  (DER variable)")
print(f"  Ed25519 firma       : {len(firma_ed):>3} bytes  (fijo)")
print(f"  P-256  clave pública: {len(p256_pub_bytes):>3} bytes  (no comprimida)")
print(f"  Ed25519 clave pública: {len(pub_raw):>3} bytes")

---
<a id='9'></a>
## 9. Ataques y consideraciones de seguridad

<a id='9-1'></a>
### 9.1 Problema del logaritmo discreto en EC (ECDLP)

Dado un punto generador $G$ de orden $n$ y un punto $Q = k \cdot G$, el **ECDLP** consiste en encontrar $k$.

- No se conoce ningún algoritmo clásico sub-exponencial **genérico** para el ECDLP.
- Los mejores algoritmos genéricos son **BSGS** y **Rho de Pollard**, con complejidad $O(\sqrt{n})$.
- Para $n \approx 2^{256}$, esto equivale a $\approx 2^{128}$ operaciones → seguridad de 128 bits.

**Comparación con factorización:**

| Problema | Mejor algoritmo | Complejidad |
|---|---|---|
| Factorización de $N$ bits | GNFS | $e^{O(N^{1/3})}$ (subexponencial) |
| ECDLP (orden $n$ de $N$ bits) | Rho de Pollard | $O(\sqrt{n}) = O(2^{N/2})$ (exponencial) |

Esto explica por qué ECC necesita claves más cortas que RSA para la misma seguridad.

<a id='9-2'></a>
### 9.2 Ataques genéricos: baby-step giant-step y Pohlig-Hellman

In [ ]:
# ── Baby-step Giant-step (BSGS) para curvas MUY pequeñas ───────────────────
# Este ataque solo es factible cuando n es pequeño.

def bsgs_ecdlp(G: ECPoint, Q: ECPoint, n: int) -> Optional[int]:
    """
    Baby-step Giant-step para resolver Q = k·G.
    Complejidad: O(sqrt(n)) tiempo y espacio.
    Solo práctico para n pequeño.
    """
    import math
    m = math.isqrt(n) + 1

    # Baby steps: tabla {j·G : j} para j = 0, ..., m-1
    baby = {}
    step = ECPoint(G.curve, None, None)  # O
    for j in range(m):
        key = (step.x, step.y)
        baby[key] = j
        step = step + G

    # Giant steps: calcular Q - i·(m·G) para i = 0, 1, ..., m-1
    mG = m * G
    neg_mG = -mG
    gamma = Q
    for i in range(m):
        if gamma.is_infinity():
            key = (None, None)
        else:
            key = (gamma.x, gamma.y)
        if key in baby:
            k = (i * m + baby[key]) % n
            return k
        gamma = gamma + neg_mG
    return None


# Demostración con curva de juguete
k_secret = 13
Q_target = k_secret * G_demo

print(f"Orden del grupo : n = {n_demo}")
print(f"Punto público   : Q = k·G con k = {k_secret}")
print(f"Q = {Q_target}")

k_found = bsgs_ecdlp(G_demo, Q_target, n_demo)
print(f"\nk encontrado por BSGS: {k_found}")
print(f"Correcto: {k_found == k_secret} ✓")
print()
print("NOTA: Para curvas reales (n ≈ 2^256), sqrt(n) ≈ 2^128 pasos → computacionalmente infactible.")

**Pohlig-Hellman:** Si el orden del grupo $n$ es un entero suave (producto de primos pequeños), el ECDLP se puede resolver eficientemente en cada subgrupo por separado (usando CRT). Esto justifica exigir que $n$ sea **primo** (o al menos tenga un factor primo grande).

<a id='9-3'></a>
### 9.3 Curvas débiles: MOV, anomalous, curvas de cofactor alto

| Vulnerabilidad | Condición | Ataque |
|---|---|---|
| **MOV** (Menezes–Okamoto–Vanstone) | $n \mid p^k - 1$ para $k$ pequeño | Reduce ECDLP a logaritmo discreto en $\mathbb{F}_{p^k}^*$ |
| **Curvas anómalas** | $|E(\mathbb{F}_p)| = p$ | ECDLP resoluble en tiempo $O(n)$ con levantamiento de Hensel (Semaev-Smart-Satoh-Araki) |
| **Cofactor alto sin verificación** | $h$ grande y no se verifica $Q \in \langle G \rangle$ | Ataques de subgrupo pequeño (small subgroup attack) |
| **Parámetros no estándar** | Curva generada ad hoc | Puede tener trapdoor (NSA dual EC DRBG) |

**Contramedida:** Usar siempre curvas estandarizadas; en ECDH con cofactor $h > 1$, calcular $h \cdot S$ antes de usar el secreto.

<a id='9-4'></a>
### 9.4 Ataque de reutilización de nonce en ECDSA (Sony PS3)

Ver Sección 7.3. La solución es:
1. Usar **RFC 6979**: generación determinista de $k$ a partir de $H(d \| z)$.
2. Usar **Ed25519** que es intrínsecamente determinista.
3. Si se usa ECDSA, agregar **diversificación con secretos aleatorios** al calcular $k$.

<a id='9-5'></a>
### 9.5 Side-channel attacks

- **Simple Power Analysis (SPA):** Si la implementación de double-and-add tiene ramas dependientes del bit del escalar, la traza de potencia revela $k$.
- **Timing attacks:** Variaciones en el tiempo de ejecución por operaciones condicionales.
- **Differential Power Analysis (DPA):** Análisis estadístico de múltiples trazas.

**Contramedidas:**
- **Montgomery ladder**: algoritmo de multiplicación escalar con tiempo constante.
- **Point blinding**: $k \cdot P = k \cdot (P + R) - k \cdot R$ para $R$ aleatorio.
- **Scalar blinding**: $k' = k + r \cdot n$ para $r$ aleatorio.
- Curve25519 / Ed25519 fueron diseñadas específicamente para ser seguras frente a side-channels.

In [ ]:
# ── Montgomery ladder: multiplicación escalar resistente a side-channel ──────

def montgomery_ladder(P: ECPoint, k: int) -> ECPoint:
    """
    Multiplicación escalar k·P con Montgomery ladder.
    Acceso constante a memoria y sin ramas dependientes del bit → resiste SPA.
    """
    R0 = ECPoint(P.curve, None, None)  # O
    R1 = P
    for bit in reversed(range(k.bit_length())):
        if (k >> bit) & 1:
            R0 = R0 + R1
            R1 = R1 + R1
        else:
            R1 = R0 + R1
            R0 = R0 + R0
    return R0


# Verificación: resultado igual al double-and-add ordinario
k_test = 42
P_test = G_demo

result_std  = k_test * P_test                   # double-and-add normal
result_ml   = montgomery_ladder(P_test, k_test) # Montgomery ladder

print(f"double-and-add    : {result_std}")
print(f"Montgomery ladder  : {result_ml}")
print(f"Resultados iguales: {result_std == result_ml} ✓")

---
<a id='10'></a>
## 10. ECC vs RSA: comparativa exhaustiva

In [ ]:
# ── Benchmarking ECC vs RSA ──────────────────────────────────────────────────
from cryptography.hazmat.primitives.asymmetric import rsa as _rsa
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding

N_ITERS = 20

def bench(label, fn):
    t0 = time.perf_counter()
    for _ in range(N_ITERS):
        fn()
    elapsed = (time.perf_counter() - t0) / N_ITERS * 1000
    print(f"  {label:<40} {elapsed:>8.2f} ms/op")

print(f"=== Benchmark ({N_ITERS} iteraciones cada uno) ===\n")

# RSA-2048
rsa2048 = _rsa.generate_private_key(public_exponent=65537, key_size=2048)
rsa4096 = _rsa.generate_private_key(public_exponent=65537, key_size=4096)
msg_b = b"benchmark message"

print("[Generación de claves]")
bench("RSA-2048 keygen",  lambda: _rsa.generate_private_key(65537, 2048))
bench("RSA-4096 keygen",  lambda: _rsa.generate_private_key(65537, 4096))
bench("ECDSA P-256 keygen", lambda: ec.generate_private_key(ec.SECP256R1()))
bench("Ed25519   keygen",  lambda: Ed25519PrivateKey.generate())
bench("X25519    keygen",  lambda: X25519PrivateKey.generate())

print("\n[Firma]")
p256k = ec.generate_private_key(ec.SECP256R1())
ed25519k = Ed25519PrivateKey.generate()
bench("RSA-2048 sign (PSS)",
      lambda: rsa2048.sign(msg_b, asym_padding.PSS(
          mgf=asym_padding.MGF1(hashes.SHA256()),
          salt_length=asym_padding.PSS.MAX_LENGTH), hashes.SHA256()))
bench("RSA-4096 sign (PSS)",
      lambda: rsa4096.sign(msg_b, asym_padding.PSS(
          mgf=asym_padding.MGF1(hashes.SHA256()),
          salt_length=asym_padding.PSS.MAX_LENGTH), hashes.SHA256()))
bench("ECDSA P-256 sign", lambda: p256k.sign(msg_b, ec.ECDSA(hashes.SHA256())))
bench("Ed25519  sign",    lambda: ed25519k.sign(msg_b))

print("\n[Verificación]")
rsa2048_sig = rsa2048.sign(msg_b, asym_padding.PSS(
    mgf=asym_padding.MGF1(hashes.SHA256()), salt_length=asym_padding.PSS.MAX_LENGTH), hashes.SHA256())
p256_sig = p256k.sign(msg_b, ec.ECDSA(hashes.SHA256()))
ed25519_sig = ed25519k.sign(msg_b)

bench("RSA-2048 verify",
      lambda: rsa2048.public_key().verify(rsa2048_sig, msg_b,
          asym_padding.PSS(mgf=asym_padding.MGF1(hashes.SHA256()),
                           salt_length=asym_padding.PSS.MAX_LENGTH), hashes.SHA256()))
bench("ECDSA P-256 verify", lambda: p256k.public_key().verify(p256_sig, msg_b, ec.ECDSA(hashes.SHA256())))
bench("Ed25519  verify",    lambda: ed25519k.public_key().verify(ed25519_sig, msg_b))

In [ ]:
# ── Tabla comparativa ECC vs RSA ─────────────────────────────────────────────

comparativa = [
    ("Problema matemático",       "Factorización de enteros",        "ECDLP"),
    ("Seguridad 128 bits",        "RSA-3072",                        "ECC-256"),
    ("Clave privada (128-bit seg)","3072 bits",                       "256 bits"),
    ("Clave pública  (128-bit seg)","3072 bits",                      "512 bits (no comp.)"),
    ("Firma (128-bit seg)",        "~384 bytes (DER)",                "~64 bytes (Ed25519)"),
    ("Subexponencial genérico",   "Sí (GNFS)",                       "No (solo $O(\\sqrt{n})$)"),
    ("Resistente a cuánticos",    "No (Shor O(log³N))",              "No (Shor O(log³n))"),
    ("Protocolo de cifrado",      "RSA-OAEP",                        "ECDH + cifrado simétrico"),
    ("Protocolo de firma",        "RSA-PSS",                         "ECDSA / EdDSA"),
    ("Perfect Forward Secrecy",   "Solo con DHE (lento)",             "ECDHE (rápido)"),
    ("Uso en TLS 1.3",            "Solo para auth (no KEX)",          "ECDHE + ECDSA/Ed25519"),
]

print(f"{'Aspecto':<35} {'RSA':<35} {'ECC':<35}")
print("-" * 105)
for aspecto, rsa_val, ecc_val in comparativa:
    print(f"{aspecto:<35} {rsa_val:<35} {ecc_val:<35}")

---
<a id='11'></a>
## 11. Buenas prácticas

### 11.1 Elección de curva

| Caso de uso | Curva recomendada | Razón |
|---|---|---|
| Firma digital (propósito general) | **Ed25519** | Determinista, rápida, claves pequeñas |
| Firma (interop. corporativa/PKI) | **P-256** | Soporte universal, FIPS 186 |
| Intercambio de clave DH | **X25519** | Resistente a side-channel por diseño |
| Blockchain / criptomonedas | **secp256k1** | Ecosistema maduro |
| Alta seguridad / gobierno | **P-384 / P-521** | Requisitos CNSA 2.0 |

### 11.2 Reglas críticas de implementación

1. **Nunca implementar la criptografía desde cero en producción.** Usar bibliotecas auditadas: `cryptography` (Python), OpenSSL, libsodium, Bouncy Castle.
2. **Validar que los puntos recibidos están en la curva** y en el subgrupo de orden primo (cofactor multiplication en X25519 ya lo hace implícitamente).
3. **En ECDSA, usar RFC 6979** (k determinista) o cambiar a Ed25519.
4. **No reutilizar claves entre ECDH y ECDSA.** Generar pares de claves distintos para cada propósito.
5. **En ECDHE, usar claves efímeras** — nunca la clave de larga duración para DH.
6. **Aplicar HKDF** sobre el secreto compartido de ECDH antes de usarlo como clave simétrica.
7. **Verificar firmas siempre** antes de procesar el mensaje, no después.
8. **Cuidado con la comparación de puntos en tiempo variable** — puede filtrar información.

### 11.3 Migraciones y resistencia cuántica

Tanto RSA como ECC son vulnerables a **algoritmos cuánticos (Shor, 1994)**. La migración hacia criptografía post-cuántica ya está en marcha:

| Algoritmo NIST PQC (2024) | Tipo | Propósito |
|---|---|---|
| **ML-KEM** (CRYSTALS-Kyber) | Retícula | Intercambio de clave (reemplaza ECDH) |
| **ML-DSA** (CRYSTALS-Dilithium) | Retícula | Firma digital (reemplaza ECDSA) |
| **SLH-DSA** (SPHINCS+) | Hash-based | Firma digital alternativa |

La estrategia de transición recomendada es **híbrida**: ECDH + ML-KEM simultáneamente, para mantener seguridad clásica mientras se adoptan los nuevos estándares.

In [ ]:
# ── Resumen de recomendaciones en código ─────────────────────────────────────

print("="*60)
print(" GUÍA RÁPIDA DE USO SEGURO DE ECC EN PYTHON")
print("="*60)

print("""
1. FIRMA DIGITAL
   from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
   key = Ed25519PrivateKey.generate()
   sig = key.sign(mensaje)
   key.public_key().verify(sig, mensaje)  # lanza InvalidSignature si falla

2. INTERCAMBIO DE CLAVE (ECDHE)
   from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
   priv = X25519PrivateKey.generate()   # nuevo par por sesión (ephemeral)
   shared = priv.exchange(peer_public_key)
   # Aplicar HKDF antes de usar shared como clave simétrica

3. CIFRADO HÍBRIDO
   X25519 (ECDHE) + HKDF-SHA256 + AES-256-GCM
   Ver sección 6.4 de esta clase

4. FIRMA INTEROPERABLE (PKI / TLS)
   from cryptography.hazmat.primitives.asymmetric import ec
   key = ec.generate_private_key(ec.SECP256R1())
   sig = key.sign(mensaje, ec.ECDSA(hashes.SHA256()))
""")
print("Documentación: https://cryptography.io/en/latest/hazmat/primitives/asymmetric/ec/")

---
<a id='12'></a>
## 12. Referencias

### Estándares y especificaciones

1. **FIPS 186-5** (2023). *Digital Signature Standard (DSS)*. NIST.  
   https://doi.org/10.6028/NIST.FIPS.186-5

2. **NIST SP 800-57 Part 1 Rev. 5** (2020). *Recommendation for Key Management*. NIST.  
   https://doi.org/10.6028/NIST.SP.800-57pt1r5

3. **RFC 6090** (2011). *Fundamental Elliptic Curve Cryptography Algorithms*. McGrew, Igoe.  
   https://www.rfc-editor.org/rfc/rfc6090

4. **RFC 6979** (2013). *Deterministic Usage of the DSA and ECDSA*. T. Pornin.  
   https://www.rfc-editor.org/rfc/rfc6979

5. **RFC 7748** (2016). *Elliptic Curves for Diffie-Hellman Key Agreement (X25519, X448)*. Langley, Hamburg, Turner.  
   https://www.rfc-editor.org/rfc/rfc7748

6. **RFC 8032** (2017). *Edwards-Curve Digital Signature Algorithm (EdDSA)*. Josefsson, Liusvaara.  
   https://www.rfc-editor.org/rfc/rfc8032

7. **SEC 2: Recommended Elliptic Curve Domain Parameters** (2010). Standards for Efficient Cryptography Group.  
   https://www.secg.org/sec2-v2.pdf

### Libros y artículos

8. **Hankerson, D., Menezes, A., Vanstone, S.** (2004). *Guide to Elliptic Curve Cryptography*. Springer.  
   Referencia clásica sobre implementación eficiente.

9. **Washington, L.C.** (2008). *Elliptic Curves: Number Theory and Cryptography* (2ª ed.). Chapman & Hall/CRC.

10. **Bernstein, D.J., Lange, T.** (2007). *Faster addition and doubling on elliptic curves*. ASIACRYPT 2007.  
    Introduce las curvas de Edwards y sus ventajas.

11. **Bernstein, D.J.** (2006). *Curve25519: New Diffie-Hellman Speed Records*. PKC 2006.  
    https://cr.yp.to/ecdh/curve25519-20060209.pdf

12. **Bernstein, D.J., Duif, N., Lange, T., Schwabe, P., Yang, B.** (2011). *High-speed high-security signatures*. CHES 2011. (Ed25519)  
    https://ed25519.cr.yp.to/ed25519-20110926.pdf

13. **Menezes, A., Okamoto, T., Vanstone, S.** (1993). *Reducing Elliptic Curve Logarithms to Logarithms in a Finite Field*. IEEE Trans. Info. Theory.  
    (Ataque MOV)

14. **Smart, N.P.** (1999). *The Discrete Logarithm Problem on Elliptic Curves of Trace One*. Journal of Cryptology.  
    (Curvas anómalas)

### Recursos en línea

15. **SafeCurves** — Criterios de seguridad para selección de curvas:  
    https://safecurves.cr.yp.to/

16. **Cryptography.io** — Documentación de la biblioteca Python:  
    https://cryptography.io/en/latest/

17. **Dan Boneh — Stanford Cryptography I & II** (Coursera):  
    https://www.coursera.org/learn/crypto

18. **CNSA 2.0 (Commercial National Security Algorithm Suite 2.0)** — NSA, 2022:  
    https://media.defense.gov/2022/Sep/07/2003071834/-1/-1/0/CSA_CNSA_2.0_ALGORITHMS_.PDF

---

> **Nota:** Esta clase hace parte de una serie de clases de Criptografía Aplicada. Los algoritmos implementados desde cero tienen **fines exclusivamente educativos** y no deben usarse en producción. Para aplicaciones reales, use siempre la biblioteca `cryptography` o equivalentes auditadas.